## Validating Datasets for Reading Files

1. Read csv and provide schema
2. Read json file
3. Convert json in one directory into respective parquet files
4. Convert csv with ',' as a separator to '|' as a separator

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Validating Datasets for Reading Files").getOrCreate()

24/03/28 18:12:48 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/03/28 18:12:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/03/28 18:12:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/03/28 18:12:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/03/28 18:12:50 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/03/28 18:12:50 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
schema = """
    order_id INT,
    order_date TIMESTAMP,
    order_customer_id INT,
    order_status STRING
"""

In [3]:
orders = spark.read.schema(schema).csv('./data/retail_db/orders/')

In [4]:
orders.show()

+--------+-------------------+-----------------+---------------+
|order_id|         order_date|order_customer_id|   order_status|
+--------+-------------------+-----------------+---------------+
|       1|2013-07-25 00:00:00|            11599|         CLOSED|
|       2|2013-07-25 00:00:00|              256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:00|            12111|       COMPLETE|
|       4|2013-07-25 00:00:00|             8827|         CLOSED|
|       5|2013-07-25 00:00:00|            11318|       COMPLETE|
|       6|2013-07-25 00:00:00|             7130|       COMPLETE|
|       7|2013-07-25 00:00:00|             4530|       COMPLETE|
|       8|2013-07-25 00:00:00|             2911|     PROCESSING|
|       9|2013-07-25 00:00:00|             5657|PENDING_PAYMENT|
|      10|2013-07-25 00:00:00|             5648|PENDING_PAYMENT|
|      11|2013-07-25 00:00:00|              918| PAYMENT_REVIEW|
|      12|2013-07-25 00:00:00|             1837|         CLOSED|
|      13|2013-07-25 00:0

In [5]:
orders = spark.read.json('./data/retail_db_json/orders/part-r-00000-990f5773-9005-49ba-b670-631286032674')

In [6]:
orders.printSchema()

root
 |-- order_customer_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_status: string (nullable = true)



In [7]:
orders.dtypes

[('order_customer_id', 'bigint'),
 ('order_date', 'string'),
 ('order_id', 'bigint'),
 ('order_status', 'string')]

In [8]:
%%bash

ls -l

total 28
-rw-r--r--  1 qasim qasim 17993 Mar 28 18:12 01 Validating Datasets for Reading Files.ipynb
-rw-r--r--  1 qasim qasim   820 Mar 28 18:05 Untitled.ipynb
drwxr-xr-x 15 qasim qasim  4096 Mar 28 18:11 data


In [9]:
%%bash

input_dir='./data/retail_db_json/'
ls -l $input_dir

total 24
drwxr-xr-x 2 qasim qasim 4096 Mar 27 22:18 categories
drwxr-xr-x 2 qasim qasim 4096 Mar 27 22:18 customers
drwxr-xr-x 2 qasim qasim 4096 Mar 27 22:18 departments
drwxr-xr-x 2 qasim qasim 4096 Mar 27 22:18 order_items
drwxr-xr-x 2 qasim qasim 4096 Mar 27 22:18 orders
drwxr-xr-x 2 qasim qasim 4096 Mar 27 22:18 products


In [10]:
# df.coalesce is added to make sure that all hte data gets written into a single file

import os

input_dir = './data/retail_db_json/'
output_dir = './data/retail_db_parquet/'
file_number = 0

for file_details in os.listdir(input_dir):
    print(f'Converting data in {os.listdir(input_dir)} folder from json to parquet')
    file_number=file_number + 1
    data_set_dir = f'{output_dir}{file_details}'
    print(data_set_dir)
    df = spark.read.json(f'{input_dir}{file_details}')
    df.coalesce(1).write.mode('overwrite').parquet(data_set_dir)
    #data_set_dir = [file_details.split('/')[-2] for file_details in {os.listdir(input_dir)}]
    #print(data_set_dir)

Converting data in ['categories', 'orders', 'order_items', 'customers', 'products', 'departments'] folder from json to parquet
./data/retail_db_parquet/categories
Converting data in ['categories', 'orders', 'order_items', 'customers', 'products', 'departments'] folder from json to parquet
./data/retail_db_parquet/orders
Converting data in ['categories', 'orders', 'order_items', 'customers', 'products', 'departments'] folder from json to parquet
./data/retail_db_parquet/order_items
Converting data in ['categories', 'orders', 'order_items', 'customers', 'products', 'departments'] folder from json to parquet
./data/retail_db_parquet/customers
Converting data in ['categories', 'orders', 'order_items', 'customers', 'products', 'departments'] folder from json to parquet
./data/retail_db_parquet/products
Converting data in ['categories', 'orders', 'order_items', 'customers', 'products', 'departments'] folder from json to parquet
./data/retail_db_parquet/departments


In [11]:
df.printSchema()

root
 |-- department_id: long (nullable = true)
 |-- department_name: string (nullable = true)



In [12]:
%%bash

input_dir='./data/retail_db_parquet/orders/'
ls -l $input_dir

total 480
-rw-r--r-- 1 qasim qasim      0 Mar 28 18:12 _SUCCESS
-rw-r--r-- 1 qasim qasim 489947 Mar 28 18:12 part-00000-f74a5615-0726-41a2-a195-0c70a6bae9ac-c000.snappy.parquet


In [14]:
orders = spark.read.parquet('./data/retail_db_parquet/orders/part-00000-f74a5615-0726-41a2-a195-0c70a6bae9ac-c000.snappy.parquet')

In [15]:
orders.show()

+-----------------+--------------------+--------+---------------+
|order_customer_id|          order_date|order_id|   order_status|
+-----------------+--------------------+--------+---------------+
|            11599|2013-07-25 00:00:...|       1|         CLOSED|
|              256|2013-07-25 00:00:...|       2|PENDING_PAYMENT|
|            12111|2013-07-25 00:00:...|       3|       COMPLETE|
|             8827|2013-07-25 00:00:...|       4|         CLOSED|
|            11318|2013-07-25 00:00:...|       5|       COMPLETE|
|             7130|2013-07-25 00:00:...|       6|       COMPLETE|
|             4530|2013-07-25 00:00:...|       7|       COMPLETE|
|             2911|2013-07-25 00:00:...|       8|     PROCESSING|
|             5657|2013-07-25 00:00:...|       9|PENDING_PAYMENT|
|             5648|2013-07-25 00:00:...|      10|PENDING_PAYMENT|
|              918|2013-07-25 00:00:...|      11| PAYMENT_REVIEW|
|             1837|2013-07-25 00:00:...|      12|         CLOSED|
|         

In [16]:
orders.printSchema()

root
 |-- order_customer_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_status: string (nullable = true)



In [17]:
orders.dtypes

[('order_customer_id', 'bigint'),
 ('order_date', 'string'),
 ('order_id', 'bigint'),
 ('order_status', 'string')]

In [26]:
# df.coalesce is added to make sure that all hte data gets written into a single file
# convert csv with ',' as a separator to '|' as a separator

import os

input_dir = './data/retail_db/'
output_dir = './data/retail_db_pipe/'
file_number = 0

for file_details in os.listdir(input_dir):
    if file_details not in 'temp' and file_details not in '.ipynb_checkpoints':
        print(f'Changing separator in {os.listdir(input_dir)} folder from comma to pipe')
        file_number=file_number + 1
        data_set_dir = f'{output_dir}{file_details}'
        print(data_set_dir)
        df = spark.read.csv(f'{input_dir}{file_details}')
        df.coalesce(1).write.mode('overwrite').csv(data_set_dir, sep='|')
        #data_set_dir = [file_details.split('/')[-2] for file_details in {os.listdir(input_dir)}]
        #print(data_set_dir)

Changing separator in ['categories', 'temp', 'orders', 'order_items', 'customers', 'products', 'departments', '.ipynb_checkpoints'] folder from comma to pipe
./data/retail_db_pipe/categories
Changing separator in ['categories', 'temp', 'orders', 'order_items', 'customers', 'products', 'departments', '.ipynb_checkpoints'] folder from comma to pipe
./data/retail_db_pipe/orders
Changing separator in ['categories', 'temp', 'orders', 'order_items', 'customers', 'products', 'departments', '.ipynb_checkpoints'] folder from comma to pipe
./data/retail_db_pipe/order_items
Changing separator in ['categories', 'temp', 'orders', 'order_items', 'customers', 'products', 'departments', '.ipynb_checkpoints'] folder from comma to pipe
./data/retail_db_pipe/customers
Changing separator in ['categories', 'temp', 'orders', 'order_items', 'customers', 'products', 'departments', '.ipynb_checkpoints'] folder from comma to pipe
./data/retail_db_pipe/products
Changing separator in ['categories', 'temp', 'order

In [24]:
orders = spark.read.csv('./data/retail_db_pipe/orders/part-00000-43b9b3c8-c56f-4cf3-a74d-da38fad38a25-c000.csv', sep='|')

In [25]:
orders.show(truncate=False)

+---+---------------------+-----+---------------+
|_c0|_c1                  |_c2  |_c3            |
+---+---------------------+-----+---------------+
|1  |2013-07-25 00:00:00.0|11599|CLOSED         |
|2  |2013-07-25 00:00:00.0|256  |PENDING_PAYMENT|
|3  |2013-07-25 00:00:00.0|12111|COMPLETE       |
|4  |2013-07-25 00:00:00.0|8827 |CLOSED         |
|5  |2013-07-25 00:00:00.0|11318|COMPLETE       |
|6  |2013-07-25 00:00:00.0|7130 |COMPLETE       |
|7  |2013-07-25 00:00:00.0|4530 |COMPLETE       |
|8  |2013-07-25 00:00:00.0|2911 |PROCESSING     |
|9  |2013-07-25 00:00:00.0|5657 |PENDING_PAYMENT|
|10 |2013-07-25 00:00:00.0|5648 |PENDING_PAYMENT|
|11 |2013-07-25 00:00:00.0|918  |PAYMENT_REVIEW |
|12 |2013-07-25 00:00:00.0|1837 |CLOSED         |
|13 |2013-07-25 00:00:00.0|9149 |PENDING_PAYMENT|
|14 |2013-07-25 00:00:00.0|9842 |PROCESSING     |
|15 |2013-07-25 00:00:00.0|2568 |COMPLETE       |
|16 |2013-07-25 00:00:00.0|7276 |PENDING_PAYMENT|
|17 |2013-07-25 00:00:00.0|2667 |COMPLETE       |
